# Image Captioning
**Captioning Base vs. Captioning + Attention**

In [ ]:
!pip install -q gensim nltk

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from nltk.translate.bleu_score import corpus_bleu
from collections import OrderedDict, Counter
from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

---
## Part A. Data Preprocessing
---

### A1. Flickr8k Download

In [ ]:
!wget -q --show-progress https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_Dataset.zip
!wget -q --show-progress https://github.com/jbrownlee/Datasets/releases/download/Flickr8k/Flickr8k_text.zip
!unzip -q Flickr8k_Dataset.zip
!unzip -q Flickr8k_text.zip -d Flickr8k_text

IMAGE_DIR = 'Flicker8k_Dataset'
TOKEN_FILE = os.path.join('Flickr8k_text', 'Flickr8k.token.txt')
TRAIN_SPLIT = os.path.join('Flickr8k_text', 'Flickr_8k.trainImages.txt')
DEV_SPLIT = os.path.join('Flickr8k_text', 'Flickr_8k.devImages.txt')
TEST_SPLIT = os.path.join('Flickr8k_text', 'Flickr_8k.testImages.txt')

print(f'Images found: {len([f for f in os.listdir(IMAGE_DIR) if f.endswith(".jpg")])}')

### A2. Caption Parsing

In [ ]:
image_captions = OrderedDict()
skipped = set()

with open(TOKEN_FILE, 'r') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        img_id, caption = line.split('\t', 1)
        filename, cap_idx = img_id.rsplit('#', 1)
        if cap_idx == '0':
            if os.path.exists(os.path.join(IMAGE_DIR, filename)):
                image_captions[filename] = caption.strip().lower()
            else:
                skipped.add(filename)

filenames = list(image_captions.keys())
captions = list(image_captions.values())
print(f'Image-caption pairs: {len(filenames)}')

### A3. Train / Val / Test Split

In [ ]:
def load_split(filepath):
    with open(filepath, 'r') as f:
        return set(line.strip() for line in f if line.strip())

train_files = load_split(TRAIN_SPLIT)
val_files = load_split(DEV_SPLIT)
test_files = load_split(TEST_SPLIT)

fname_to_idx = {fname: i for i, fname in enumerate(filenames)}
train_idx = [fname_to_idx[f] for f in filenames if f in train_files]
val_idx = [fname_to_idx[f] for f in filenames if f in val_files]
test_idx = [fname_to_idx[f] for f in filenames if f in test_files]

splits = {'train': train_idx, 'val': val_idx, 'test': test_idx}
print(f'Train: {len(train_idx)}, Val: {len(val_idx)}, Test: {len(test_idx)}')

### A4. ResNet-50 Feature Map (2048×7×7)

In [ ]:
resnet = torchvision.models.resnet50(weights='DEFAULT')
feature_extractor = nn.Sequential(
    resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
    resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4
)
feature_extractor = feature_extractor.to(device)
feature_extractor.eval()

preprocess = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print('Extracting feature maps (2048x7x7)...')
img_feats = []
with torch.no_grad():
    for fname in tqdm(filenames):
        img = Image.open(os.path.join(IMAGE_DIR, fname)).convert('RGB')
        img_tensor = preprocess(img).unsqueeze(0).to(device)
        feat = feature_extractor(img_tensor).squeeze(0).cpu()
        img_feats.append(feat.half())

img_feats = torch.stack(img_feats)
print(f'Feature maps: {img_feats.shape} ({img_feats.dtype})')

del resnet, feature_extractor  # free memory
torch.cuda.empty_cache() if torch.cuda.is_available() else None

### A5. GloVe Vocabulary

In [ ]:
import gensim.downloader as api

print('Loading GloVe...')
glove = api.load('glove-wiki-gigaword-300')
print(f'GloVe loaded: {len(glove)} words, {glove.vector_size}d')

In [ ]:
word_counts = Counter()
for cap in captions:
    word_counts.update(cap.split())

PAD_IDX, UNK_IDX = 0, 1
word2idx = {'<pad>': PAD_IDX, '<unk>': UNK_IDX}
vectors = [np.zeros(300), np.random.randn(300) * 0.01]

found, not_found = 0, 0
for word in word_counts:
    if word in glove:
        word2idx[word] = len(word2idx)
        vectors.append(glove[word])
        found += 1
    else:
        not_found += 1

glove_vectors = torch.FloatTensor(np.stack(vectors))
idx2word = {v: k for k, v in word2idx.items()}
print(f'Vocab: {len(word2idx)} (hit: {found}, miss: {not_found})')

del glove  # free memory

### A6. Caption Tokenization + Special Tokens

In [ ]:
MAX_LEN = 32

def caption_to_ids(caption, word2idx, max_len=MAX_LEN):
    tokens = caption.split()
    ids = [word2idx.get(w, UNK_IDX) for w in tokens]
    ids = ids[:max_len]
    length = len(ids)
    ids = ids + [PAD_IDX] * (max_len - length)
    return ids, length

all_ids, all_lens = [], []
for cap in captions:
    ids, length = caption_to_ids(cap, word2idx)
    all_ids.append(ids)
    all_lens.append(length)

caption_ids = torch.tensor(all_ids, dtype=torch.long)
caption_lengths = torch.tensor(all_lens, dtype=torch.long)

# <start>, <end> tokens
START_IDX = len(word2idx)
END_IDX = len(word2idx) + 1
word2idx['<start>'] = START_IDX
word2idx['<end>'] = END_IDX
idx2word[START_IDX] = '<start>'
idx2word[END_IDX] = '<end>'
VOCAB_SIZE = len(word2idx)

extra = torch.randn(2, 300) * 0.01
glove_vectors = torch.cat([glove_vectors, extra], dim=0)

print(f'Caption IDs: {caption_ids.shape}')
print(f'Vocab: {VOCAB_SIZE} (<start>={START_IDX}, <end>={END_IDX})')

In [ ]:
# Teacher forcing sequences
MAX_SEQ = 34

all_input_seqs, all_target_seqs, all_seq_lengths = [], [], []
for i in range(len(caption_ids)):
    cap = caption_ids[i][:caption_lengths[i]].tolist()
    inp = [START_IDX] + cap
    tgt = cap + [END_IDX]
    length = len(inp)
    inp = (inp + [PAD_IDX] * MAX_SEQ)[:MAX_SEQ]
    tgt = (tgt + [PAD_IDX] * MAX_SEQ)[:MAX_SEQ]
    all_input_seqs.append(inp)
    all_target_seqs.append(tgt)
    all_seq_lengths.append(length)

input_seqs = torch.tensor(all_input_seqs, dtype=torch.long)
target_seqs = torch.tensor(all_target_seqs, dtype=torch.long)
seq_lengths = torch.tensor(all_seq_lengths, dtype=torch.long)

i = splits['train'][0]
inp_words = [idx2word[t.item()] for t in input_seqs[i][:seq_lengths[i]]]
tgt_words = [idx2word[t.item()] for t in target_seqs[i][:seq_lengths[i]]]
print(f'Example:')
print(f'  Input:  {inp_words[:8]}...')
print(f'  Target: {tgt_words[:8]}...')

---
## Part B. Model Training & Evaluation
---

### B1. Dataset & DataLoader

In [ ]:
class CaptionDataset(Dataset):
    def __init__(self, img_feats, input_seqs, target_seqs, seq_lengths, indices):
        self.img_feats = img_feats[indices]
        self.input_seqs = input_seqs[indices]
        self.target_seqs = target_seqs[indices]
        self.seq_lengths = seq_lengths[indices]

    def __len__(self):
        return len(self.input_seqs)

    def __getitem__(self, idx):
        return (
            self.img_feats[idx].float(),
            self.input_seqs[idx],
            self.target_seqs[idx],
            self.seq_lengths[idx]
        )

BATCH_SIZE = 64
train_ds = CaptionDataset(img_feats, input_seqs, target_seqs, seq_lengths, splits['train'])
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
print(f'Train: {len(train_ds)}, {len(train_loader)} batches')

### B2. Model Definition

**Captioning Base (no attention)**
```
Image: (2048,7,7) → avgpool → (2048) → FC → s₀ (512)
Decoder: s₀ + embed(<start>) → LSTM → "a" → "dog" → ... → <end>
```

**Captioning + Attention**
```
Image: (2048,7,7) → reshape → 49 spatial vectors h_{i,j} (2048)
  s₀, c₀ = FC(mean(h_{i,j}))
  each timestep:
    eᵢ = v'ᵢ · q' / √d                  ← similarity score
    αₜ = softmax(e)                       ← attention weight
    v̂ₜ = Σ αᵢvᵢ → FC → (300)             ← attended feature
    LSTM([embed(word); v̂ₜ], s_{t-1}) → sₜ → next word
```

In [ ]:
class CaptionBase(nn.Module):
    def __init__(self, glove_vectors, hidden_dim=512, vocab_size=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.embed = nn.Embedding.from_pretrained(glove_vectors, padding_idx=0)
        self.embed.weight.requires_grad = True
        self.init_s = nn.Linear(2048, hidden_dim)   # → s₀
        self.init_c = nn.Linear(2048, hidden_dim)   # → c₀ (LSTM cell state)
        self.lstm = nn.LSTMCell(300, hidden_dim)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def forward(self, img_feat, input_seq):
        B = img_feat.size(0)
        v = img_feat.mean(dim=[2, 3])               # avgpool → (B, 2048)
        s = self.init_s(v)                           # s₀
        c = self.init_c(v)                           # c₀
        embeddings = self.embed(input_seq)           # (B, T, 300)
        outputs = []
        for t in range(input_seq.size(1)):
            s, c = self.lstm(embeddings[:, t], (s, c))  # s_t
            outputs.append(self.fc_out(s))              # y_t
        return torch.stack(outputs, dim=1)

    @torch.no_grad()
    def generate(self, img_feat, max_len=20):
        self.eval()
        v = img_feat.mean(dim=[2, 3])
        s = self.init_s(v)
        c = self.init_c(v)
        word_idx = START_IDX
        generated = []
        for _ in range(max_len):
            x = self.embed(torch.tensor([[word_idx]], device=img_feat.device)).squeeze(1)
            s, c = self.lstm(x, (s, c))
            word_idx = self.fc_out(s).argmax(1).item()
            if word_idx == END_IDX:
                break
            generated.append(word_idx)
        return ' '.join(idx2word.get(w, '<unk>') for w in generated)

In [ ]:
class CaptionAttention(nn.Module):
    def __init__(self, glove_vectors, hidden_dim=512, att_dim=256, vocab_size=None):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.att_dim = att_dim
        self.embed = nn.Embedding.from_pretrained(glove_vectors, padding_idx=0)
        self.embed.weight.requires_grad = True

        self.init_s = nn.Linear(2048, hidden_dim)    # image feature -> s₀
        self.init_c = nn.Linear(2048, hidden_dim)    # image feature -> c₀

        # Attention
        self.W_v = nn.Linear(2048, att_dim)          # spatial feature projection
        self.W_s = nn.Linear(hidden_dim, att_dim)    # previous hidden state projection

        self.ctx_proj = nn.Linear(2048, 300)         # context vector -> 300d
        self.lstm = nn.LSTMCell(600, hidden_dim)     # [word embedding(300); context(300)]
        self.fc_out = nn.Linear(hidden_dim, vocab_size)

    def _attention(self, h_proj, s, spatial):
        #① 이전 시점의 decoder hidden state s_{t-1}를 attention 차원(att_dim)으로 투영한다.
        #이 벡터는 각 spatial feature와 비교하기 위한 query 역할을 한다.
        #Shape: (B, 512) -> (B, 256)
        s_proj = self.W_s(s)
        #② 각 spatial location마다 query와의 내적으로 유사도를 계산해 attention score를 만든다.
        #즉, 49개 위치 각각이 현재 문맥과 얼마나 관련 있는지 측정하는 단계다.
        #Shape: (B, 49, 256), (B, 1, 256) -> (B, 49)
        e = (h_proj * s_proj.unsqueeze(1)).sum(dim=2)
        #③ attention score를 sqrt(att_dim)으로 나누어 softmax가 너무 뾰족해지지 않도록 안정화한다.
        #softmax는 지수함수를 이용하기에 너무 크면 값이 치우칠 수 있는데, 이 스케일링을 통해 gradient가 불안정해지는 것을 줄여준다.
        #Shape: (B, 49) -> (B, 49)
        e = e / (self.att_dim ** 0.5)
        #④ score를 softmax로 정규화해 49개 spatial 위치에 대한 attention weight를 얻는다.
        #각 값은 0~1 사이의 확률이며 합은 1이 된다.
        #Shape: (B, 49) -> (B, 49)
        a = F.softmax(e, dim=1)
        #⑤ attention weight로 spatial feature를 가중합해 context vector ctx를 만든다.
        #중요한 위치는 더 크게 반영되고 덜 중요한 위치는 약하게 반영된다.
        #Shape: (B, 49, 1) * (B, 49, 2048) -> (B, 2048)
        ctx = (a.unsqueeze(2) * spatial).sum(dim=1)
        return ctx, a

    def forward(self, img_feat, input_seq):
        B = img_feat.size(0)
        #⑥ ResNet feature map을 7x7 공간 격자 기준으로 펼쳐서 (49, 2048) sequence로 바꾼다.
        #이렇게 해야 attention이 각 spatial location을 개별적으로 볼 수 있다.
        #Shape: (B, 2048, 7, 7) -> (B, 49, 2048)
        spatial = img_feat.view(B, 2048, -1).permute(0, 2, 1).contiguous()
        #⑦ 각 spatial feature를 attention key용 차원(att_dim)으로 투영한다.
        #query(s_proj)와 같은 차원으로 맞춰 내적을 계산할 수 있게 만든다.
        #Shape: (B, 49, 2048) -> (B, 49, 256)
        h_proj = self.W_v(spatial)
        #⑧ 모든 spatial location의 feature를 평균내어 이미지 전체의 전역 요약 정보를 생성한다.
        #이 값은 decoder 초기 state를 만드는 데 사용된다.
        #Shape: (B, 49, 2048) -> (B, 2048)
        v_mean = spatial.mean(dim=1)
        #⑨ 전역 이미지 요약 벡터를 이용해 decoder의 초기 hidden state s_0와 initial cell state c_0를 생성한다.
        #이 초기값은 caption generation을 시작할 때의 첫 문맥 역할을 한다.
        #Shape: (B, 2048) -> (B, 512), (B, 2048) -> (B, 512)
        s = self.init_s(v_mean)
        c = self.init_c(v_mean)
        embeddings = self.embed(input_seq)
        outputs = []
        for t in range(input_seq.size(1)):
            #⑩ 현재 decoder state s를 기준으로 attention을 다시 계산해 attention context ctx를 얻는다.
            #반복되는 decoder loop 안에서 매 time step마다 다른 spatial location에 주목할 수 있도록 호출된다.
            #Shape: h_proj(B, 49, 256), s(B, 512), spatial(B, 49, 2048) -> (B, 2048)
            ctx, _ = self._attention(h_proj, s, spatial)
            #⑪ context vector를 단어 embedding 차원(300)으로 투영한다.
            #이후 현재 단어 embedding과 concat하기 위해 차원을 맞추는 단계다.
            #Shape: (B, 2048) -> (B, 300)
            ctx_300 = self.ctx_proj(ctx)
            #⑫ 현재 시점의 단어 embedding과 attention context를 이어 붙여 LSTM 입력을 만든다.
            #즉, 현재 시점 입력 단어의 embedding embeddings[:, t]와 이미지에서 주목한 정보 ctx_300을 함께 사용한다.
            #Shape: (B, 300) + (B, 300) -> (B, 600)
            lstm_input = torch.cat([embeddings[:, t], ctx_300], dim=1)
            #⑬ LSTMCell이 이전 hidden/cell state와 현재 입력을 받아 새로운 decoder state를 갱신한다.
            #현재 입력과 이전 (hidden state, cell state)를 받아 새로운 (hidden state, cell state)를 만든다
            #Shape: (B, 600), ((B, 512), (B, 512)) -> ((B, 512), (B, 512))
            s, c = self.lstm(lstm_input, (s, c))
            #⑭ 갱신된 hidden state s를 vocabulary logits으로 변환해 다음 단어 분포를 만든다.
            #학습 시에는 CrossEntropyLoss에 들어가고, 생성 시에는 argmax로 다음 단어를 고른다.
            #Shape: (B, 512) -> (B, vocab_size)
            outputs.append(self.fc_out(s))
        return torch.stack(outputs, dim=1)

    @torch.no_grad()
    def generate(self, img_feat, max_len=20, return_attention=False):
        self.eval()
        B = img_feat.size(0)
        spatial = img_feat.view(B, 2048, -1).permute(0, 2, 1).contiguous()
        h_proj = self.W_v(spatial)
        v_mean = spatial.mean(dim=1)
        s = self.init_s(v_mean)
        c = self.init_c(v_mean)
        word_idx = START_IDX
        generated, attentions = [], []
        for _ in range(max_len):
            ctx, a = self._attention(h_proj, s, spatial)
            ctx_300 = self.ctx_proj(ctx)
            x = self.embed(torch.tensor([[word_idx]], device=img_feat.device)).squeeze(1)
            lstm_input = torch.cat([x, ctx_300], dim=1)
            s, c = self.lstm(lstm_input, (s, c))
            word_idx = self.fc_out(s).argmax(1).item()
            if word_idx == END_IDX:
                break
            generated.append(word_idx)
            attentions.append(a.squeeze(0).cpu().numpy())
        words = [idx2word.get(w, '<unk>') for w in generated]
        caption = ' '.join(words)
        if return_attention:
            return caption, words, attentions
        return caption

### B3. Training & Evaluation Functions

In [ ]:
def train_model(model, num_epochs=30, lr=1e-3):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)
    train_losses = []
    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        for img_feat, inp_seq, tgt_seq, _ in train_loader:
            img_feat = img_feat.to(device)
            inp_seq = inp_seq.to(device)
            tgt_seq = tgt_seq.to(device)
            logits = model(img_feat, inp_seq)
            loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_seq.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        train_losses.append(epoch_loss / len(train_loader))
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Epoch {epoch+1:2d}/{num_epochs}  loss: {train_losses[-1]:.4f}')
    return train_losses

In [ ]:
@torch.no_grad()
def evaluate_bleu(model, split_indices):
    model.eval()
    references, hypotheses = [], []
    for img_idx in split_indices:
        feat = img_feats[img_idx].float().unsqueeze(0).to(device)
        generated = model.generate(feat)
        hyp = generated.split()
        refs = [captions[img_idx].split()]
        references.append(refs)
        hypotheses.append(hyp)
    b1 = corpus_bleu(references, hypotheses, weights=(1, 0, 0, 0)) * 100
    b4 = corpus_bleu(references, hypotheses, weights=(0.25, 0.25, 0.25, 0.25)) * 100
    print(f'  BLEU-1: {b1:.1f}  BLEU-4: {b4:.1f}')
    return b1, b4

def show_captions(model, img_indices, title=''):
    model.eval()
    n = len(img_indices)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1: axes = [axes]
    for ax, idx in zip(axes, img_indices):
        feat = img_feats[idx].float().unsqueeze(0).to(device)
        generated = model.generate(feat)
        img = Image.open(os.path.join(IMAGE_DIR, filenames[idx]))
        ax.imshow(img)
        ax.set_title(f'GT: {captions[idx][:50]}...\nGen: {generated[:50]}', fontsize=9, pad=8)
        ax.axis('off')
    if title: fig.suptitle(title, fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

### B4. Attention Visualization

In [ ]:
def show_attention_over_words(model, img_idx):
    model.eval()
    feat = img_feats[img_idx].float().unsqueeze(0).to(device)
    caption, words, attentions = model.generate(feat, return_attention=True)
    img = Image.open(os.path.join(IMAGE_DIR, filenames[img_idx])).resize((224, 224))

    n_words = min(len(words), 8)
    if n_words == 0:
        print(f'No words generated for {filenames[img_idx]}')
        return

    fig, axes = plt.subplots(2, n_words, figsize=(2.5 * n_words, 5.5))
    if n_words == 1: axes = axes.reshape(2, 1)

    for t in range(n_words):
        att_map = attentions[t].reshape(7, 7)
        att_up = np.array(Image.fromarray(att_map).resize((224, 224), Image.BILINEAR))
        axes[0, t].imshow(img)
        axes[0, t].imshow(att_up, cmap='hot', alpha=0.5)
        axes[0, t].set_title(f'"{words[t]}"', fontsize=11, fontweight='bold')
        axes[0, t].axis('off')
        axes[1, t].imshow(att_map, cmap='hot', interpolation='nearest')
        axes[1, t].axis('off')

    fig.suptitle(f'GT: "{captions[img_idx][:60]}"\nGen: "{caption}"', fontsize=11, y=1.05)
    plt.tight_layout()
    plt.show()

### B5. Experiment 1: Captioning Base (no attention)

In [ ]:
model_base = CaptionBase(glove_vectors, hidden_dim=512, vocab_size=VOCAB_SIZE).to(device)

print('=== Before Training ===')
show_captions(model_base, splits['test'][20:23], 'Captioning Base (before training)')

In [ ]:
print('--- Captioning Base Training ---')
base_losses = train_model(model_base, num_epochs=30)

In [ ]:
print('=== Captioning Base: After Training ===')
show_captions(model_base, splits['test'][20:25], 'Captioning Base (after training)')
print('\nBLEU (test):')
_ = evaluate_bleu(model_base, splits['test'])

### B6. Experiment 2: Captioning + Attention

In [ ]:
model_att = CaptionAttention(glove_vectors, hidden_dim=512, att_dim=256,
                             vocab_size=VOCAB_SIZE).to(device)

print('=== Attention: Before Training ===')
show_attention_over_words(model_att, splits['test'][20])

In [ ]:
print('--- Captioning + Attention Training ---')
att_losses = train_model(model_att, num_epochs=30)

In [ ]:
print('=== Captioning + Attention: After Training ===')
show_captions(model_att, splits['test'][20:25], 'Captioning + Attention (after training)')
print('\nBLEU (test):')
_ = evaluate_bleu(model_att, splits['test'])

In [ ]:
print('=== Attention Visualization (after training) ===')
for idx in splits['test'][20:25]:
    show_attention_over_words(model_att, idx)
    print()

### B7. Comparison

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(base_losses, label='Captioning Base')
plt.plot(att_losses, label='Captioning + Attention')
plt.xlabel('Epoch'); plt.ylabel('Train Loss')
plt.title('Training Loss')
plt.legend(); plt.grid(alpha=0.3)
plt.show()

In [ ]:
print('=== BLEU Comparison (test set) ===\n')
print('Captioning Base:')
_ = evaluate_bleu(model_base, splits['test'])
print('\nCaptioning + Attention:')
_ = evaluate_bleu(model_att, splits['test'])

In [ ]:
print('=== Caption Comparison ===')
for idx in splits['test'][20:25]:
    feat = img_feats[idx].float().unsqueeze(0).to(device)
    cap_base = model_base.generate(feat)
    cap_att = model_att.generate(feat)
    print(f'{filenames[idx]}')
    print(f'  GT:        {captions[idx]}')
    print(f'  Base:      {cap_base}')
    print(f'  Attention: {cap_att}')
    print()